# TP Integrador Neo4j, MongoDB y Redis
Ghizzo Agustín
Meza Joaquín

## Celda de importaciones y conexiones

In [1]:
import os 
import redis 
import matplotlib.pyplot as plt
from pymongo import MongoClient 
from neo4j import GraphDatabase

# Conexiones
MONGO_HOST = "mongo"
REDIS_HOST = "redis"
NEO4J_HOST = "neo4j"

MONGO_USER = os.getenv("MONGO_INITDB_ROOT_USERNAME", "admin")
MONGO_PASS = os.getenv("MONGO_INITDB_ROOT_PASSWORD", "admin123")
REDIS_PASS = os.getenv("REDIS_PASSWORD", "redis123")
NEO4J_PASS = os.getenv("NEO4J_PASSWORD", "neo4j123")

MONGO_URI = f"mongodb://{MONGO_USER}:{MONGO_PASS}@{MONGO_HOST}:27017/?authSource=admin"
NEO4J_URI = f"bolt://{NEO4J_HOST}:7687"

# Inicialización de conexiones 

try:
    mongo_client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=3000)
    db_mongo = mongo_client["tpIntegrador"]
    mongo_client.admin.command('ping')
    print("Conexión a MongoDB exitosa")

    redis_client = redis.Redis(host=REDIS_HOST, port=6379, password=REDIS_PASS, decode_responses=True)
    redis_client.ping()
    print("Conexión exitosa a Redis")

    neo4j_driver = GraphDatabase.driver(NEO4J_URI, auth=("neo4j", NEO4J_PASS))
    with neo4j_driver.session() as session:
        session.run("RETURN 1")
    print("Conexión a Neo4j exitosa")

except Exception as e:
    print({e})


# Se instancia fecha referencia en formato ISO

FECHA_REFERENCIA = "2026-10-01"

# Carga de las bases de datos




Conexión a MongoDB exitosa
Conexión exitosa a Redis
Conexión a Neo4j exitosa


In [2]:
#celda provisoria de importaciones
import sys
import os

sys.path.append(os.path.abspath("../python"))
sys.path.append("/work")

print("1\. Directorio de trabajo actual:", os.getcwd()) 
print("2\. Archivos/Carpetas aquí:", os.listdir('.')) 
if os.path.exists('/work'): 
    print("3\. Contenido de /work:", os.listdir('/work'))

1\. Directorio de trabajo actual: /work
2\. Archivos/Carpetas aquí: ['.ipynb_checkpoints', 'src', 'tpIntegrador.ipynb']
3\. Contenido de /work: ['.ipynb_checkpoints', 'src', 'tpIntegrador.ipynb']


# EJERCICIO 3 - 12 CONSULTAS 

1. Usuarios que asistieron a algún evento de "Los Cóndores Eléctricos".
2. Amigos de Juan López que siguen algún artista que también sigue Juan. Mostrar el nombre del amigo y el artista.
3. Recomendar a Juan artistas que siguen sus amigos y que él no sigue, ordenados por cantidad de amigos que los siguen.
4. Para los artistas del punto anterior, listar sus eventos futuros con lugar, ciudad, fecha, precio y entradas disponibles.
5. Artistas similares a los que sigue Lucía Romero y que ella no sigue.
6. Eventos futuros en La Plata con precio menor a $40.000.
7. Cantidad de eventos futuros por género.
8. Total gastado por cada usuario en compras confirmadas, ordenado de mayor a menor.
9. Entradas vendidas por evento, incluyendo los eventos sin ventas.
10. Usuarios conectados en este momento.
11. Reservas vigentes: usuario, evento, cantidad y tiempo restante antes de que venzan.
12. Los 3 géneros más buscados.

# Importaciones y limpieza las bases de datos 

In [3]:
# provisorio:

from src.carga_inicial import insert_mongo, insert_neo4j, insert_redis


#Limpio las bases de datos 

for col in ["usuarios", "artistas", "lugares", "eventos", "compras"]:
    db_mongo[col].delete_many({})

redis_client.flushdb()

with neo4j_driver.session() as s:
    s.run("MATCH (n) DETACH DELETE n")

print("Bases de datos limpias")

#Carga de bases de datos

insert_mongo(db_mongo)
insert_redis(redis_client)
insert_neo4j(neo4j_driver)

SyntaxError: invalid syntax (carga_inicial.py, line 7)

In [31]:
print("Colecciones:", db_mongo.list_collection_names()) 
print("Usuarios:", list(db_mongo.usuarios.find())) 
print("Eventos:", list(db_mongo.eventos.find())) 
print("Compras:", list(db_mongo.compras.find()))

Colecciones: ['compras', 'lugares', 'eventos', 'usuarios', 'artistas']
Usuarios: [{'_id': ObjectId('6ad7bfa31a3fb182dd419b52'), 'usuario_id': 1, 'nombre': 'Sofía Martínez', 'email': 'sofia.martinez@example.com', 'ciudad': 'La Plata', 'amigos': [2, 3], 'sigue': [2, 5]}, {'_id': ObjectId('6ad7bfa31a3fb182dd419b53'), 'usuario_id': 2, 'nombre': 'Juan López', 'email': 'juan.lopez@example.com', 'ciudad': 'La Plata', 'amigos': [1, 4, 5], 'sigue': [1, 3]}, {'_id': ObjectId('6ad7bfa31a3fb182dd419b54'), 'usuario_id': 3, 'nombre': 'Lucía Romero', 'email': 'lucia.romero@example.com', 'ciudad': 'CABA', 'amigos': [1, 6], 'sigue': [2, 4]}, {'_id': ObjectId('6ad7bfa31a3fb182dd419b55'), 'usuario_id': 4, 'nombre': 'Martín Díaz', 'email': 'martin.diaz@example.com', 'ciudad': 'Rosario', 'amigos': [2], 'sigue': [3, 6]}, {'_id': ObjectId('6ad7bfa31a3fb182dd419b56'), 'usuario_id': 5, 'nombre': 'Valentina Ruiz', 'email': 'valentina.ruiz@example.com', 'ciudad': 'Córdoba', 'amigos': [2], 'sigue': [1, 4, 6]}, {'

# 1) Usuarios que asistieron a algún evento de "Los Cóndores Eléctricos"

In [ ]:
artista = db_mongo.artistas.find_one({"nombre":"Los Cóndores Eléctricos"})
artista_id = artista["artista_id"] 

eventos_pasados = db_mongo.eventos.find({"artistas":artista_id, "fecha":{"$lt":FECHA_REFERENCIA} })
eventos_ids = [e["evento_id"] for e in eventos_pasados]

compras_confirmadas = db_mongo.compras.find({"evento_id": {"$in": eventos_ids }, "estado": "confirmada"})

#Quitando duplicados
usuarios_ids = list(set(c["usuario_id"] for c in compras_confirmadas))

usuarios = db_mongo.usuarios.find({"usuario_id": {"$in": usuarios_ids}})

#Imprimiendo

print("Usuarios que asistieron a al menos un evento de 'Los cóndores Eléctricos': ")
for usu in usuarios:
    print(f" {usu['nombre']} (ID: {usu['usuario_id']})")


# 2) Amigos de Juan López que siguen algún artista que también sigue Juan. Mostrar el nombre del amigo y el artista.

In [ ]:
query_2 = """
MATCH (juan:Usuario {nombre: "Juan López"})-[:AMIGO_DE]-(amigo:Usuario), (juan)-[:SIGUE]->(artista:Artista), (amigo)-[:SIGUE]->(artista)
RETURN amigo.nombre, artista.nombre
"""

with neo4j_driver.session() as session:
    for r in session.run(query_2):
        print(f"Con el amigo: {r['amigo.nombre']} comparten al artista: {r['artista.nombre']}")

# 3) Recomendar a Juan artistas que siguen sus amigos y que él no sigue, ordenados por cantidad de amigos que los siguen.

In [ ]:
query_3= """
MATCH (juan:Usuario {nombre: "Juan López"})-[:AMIGO_DE]-(amigo:Usuario)-[:SIGUE]->(artista:Artista)
WITH juan, artista, amigo, EXISTS((juan)-[:SIGUE]->(artista)) AS juan_ya_lo_sigue
WHERE NOT juan_ya_lo_sigue
RETURN artista.nombre, COUNT(amigo) AS cantidad_amigos
ORDER BY cantidad_amigos DESC
"""

with neo4j_driver.session() as session:
    for r in session.run(query_3):
        print(f"Artista recomendado: {r['artista.nombre']} , seguido por {r['cantidad_amigos']} de tus amigos")


# 4) Para los artistas del punto anterior, listar sus eventos futuros con lugar, ciudad, fecha, precio y entradas disponibles.

In [ ]:
print()
print("Inciso 4: ")

with neo4j_driver.session() as session:
    nombres_recomendados = [r['artista.nombre'] for r in session.run(query_3)]

artistas_mongo = db_mongo.artistas.find({"nombre": {"$in": nombres_recomendados}})
ids_recomendados = [artista['artista_id'] for artista in artistas_mongo]

eventos_futuros_recomendados = db_mongo.eventos.find({
    'artistas':{'$in':ids_recomendados},
    'fecha':{'$gt':FECHA_REFERENCIA},
    'estado':'activo'
})
for e in eventos_futuros_recomendados:
    lugar = db_mongo.lugares.find_one({
        'lugar_id':e['lugar_id']
    })
    print(f"Evento recomendado: {e['nombre']} el {e['fecha']} a ${e['precio']} en {lugar['ciudad']}")



# 5) Artistas similares a los que sigue Lucía Romero y que ella no sigue.

In [ ]:
print()
print("Inciso 5: ")
query_5 = """
MATCH (lucia:Usuario {nombre: "Lucía Romero"})-[:SIGUE]->(a:Artista)-[:SIMILAR_A]-(s:Artista)
WHERE NOT (lucia)-[:SIGUE]->(s)
RETURN s.nombre AS similar, collect(DISTINCT a.nombre) AS porque_sigue_a
"""
with neo4j_driver.session() as session:
    for r in session.run(query_5):
        print(f"{r['similar']} (similar a {', '.join(r['porque_sigue_a'])})")

# 6) Eventos futuros en La Plata con precio menor a $40.000.

In [ ]:
print()
print('Inciso 6: ')
lugares_lp = [l["lugar_id"] for l in db_mongo.lugares.find({"ciudad": "La Plata"})]
eventos_futuros_baratos = db_mongo.eventos.find({
    'lugar_id':{'$in': lugares_lp},
    'precio':{'$lt':40000},
    'estado':'activo',
    'fecha': {'$gt': FECHA_REFERENCIA}
})

for evento in eventos_futuros_baratos:
    lugar = db_mongo.lugares.find_one({'lugar_id':evento['lugar_id']})
    print(f"Te recomendamos asistir a {evento['nombre']} por solo ${evento['precio']} el día: {evento['fecha']} en {lugar['nombre']}")


# 7) Cantidad de eventos futuros por género.

In [ ]:
print()
print('Inciso 7')
#Uso una agreggation pipeline

pipeline_7 = [
    {'$match': {'fecha': {'$gte':FECHA_REFERENCIA}}},
    {'$group': {'_id': '$genero', 'cantidad_eventos':{'$sum':1}}}
]

ev_futuros_por_genero = db_mongo.eventos.aggregate(pipeline_7)


for resultado in ev_futuros_por_genero:
    print(f"{resultado['_id']}: {resultado['cantidad_eventos']} eventos")


# 8) Total gastado por cada usuario en compras confirmadas, ordenado de mayor a menor.

In [ ]:
print()
print('Inciso 8')

pipeline_8 = [
    {'$match':{'estado':'confirmada'}},
    {'$group':{'_id': '$usuario_id','total':{'$sum':'$total'}}},
    {'$sort':{'total': -1}}
]

for r in db_mongo.compras.aggregate(pipeline_8):
    u = db_mongo.usuarios.find_one({'usuario_id': r['_id']})
    print(f"{u['nombre']}: ${r['total']:,}")

##  9. Entradas vendidas por evento, incluyendo los eventos sin ventas.

In [ ]:
print("Inciso 9: Entradas vendidas por evento")

#Traigo todos los eventos
todos_los_eventos = db_mongo.eventos.find({})

for evento in todos_los_eventos:
    compras_del_evento = db_mongo.compras.find({
        "evento_id": evento["evento_id"],
        "estado": "confirmada"
    })

    total_entradas = sum([compra["cantidad"] for compra in compras_del_evento])
    
    print(f"Evento: {evento['nombre']} - Entradas vendidas: {total_entradas}")

## 10. Usuarios conectados en este momento.

In [ ]:
#provisorio
claves_activas = redis_client.keys('*') 

if not claves_activas:
    print("No hay ningún usuario conectado en este momento.")

for clave in claves_activas:
    valor = redis_client.get(clave)
    
    print(f"Registro activo en Redis -> Clave: {clave}, Valor: {valor}")

## 11. Reservas vigentes: usuario, evento, cantidad y tiempo restante antes de que venzan.

In [ ]:
claves_reservas = redis_client.keys('reserva:*')


if not claves_reservas:
    print("No hay reservas vigentes en este momento.")

#redis_client.setex("reserva:usuario_2:evento_1", 300, "cantidad: 2")

for clave in claves_reservas:
    # 2. Le preguntamos a Redis los segundos exactos que le quedan de vida a esa llave
    tiempo_restante = redis_client.ttl(clave)
    
    # 3. Traemos los datos guardados (usuario, evento, cantidad)
    datos_reserva = redis_client.get(clave)
    
    print(f"Reserva: {clave} | Datos: {datos_reserva} | Vence en: {tiempo_restante} segundos")

## 12. Los 3 géneros más buscados.

In [ ]:
top3 = redis_client.zrevrange("busquedas:generos", 0, 2, withscores=True)
for genero, puntaje in top3:
    print(f"{genero}: {int(puntaje)} búsquedas")

## Ejercicio 4 - Estadísticas y gráficos

Generar las siguientes estadísticas, mostrando el valor para cada elemento (no solo el máximo):
• Cantidad de seguidores por artista.
• Recaudación por evento (compras confirmadas).
• Entradas vendidas por lugar.
• Cantidad de búsquedas por género.
Representar al menos dos de estas estadísticas con gráficos hechos con matplotlib. Los datos de los gráficos deben obtenerse consultando las bases, no desde las listas de Python del dataset.

In [ ]:
# Función reutilizada de un proyecto anterior, adaptada para jupyterlab:

def generar_grafico_generico(datos_x, datos_y, tipo_grafico="bar", titulo="", x_etiqueta="", y_etiqueta="", **kwargs):
    """
    Crea un gráfico de Matplotlib según los argumentos recibidos y lo renderiza en Streamlit
    """
    fig, ax = plt.subplots(figsize=kwargs.get('figsize', (10, 6)))
    if tipo_grafico == 'bar':
        ax.bar(datos_x, datos_y, color=kwargs.get('color', '#4db6ac'))
        plt.xticks(rotation=45)
    elif tipo_grafico == 'line':
        ax.plot(datos_x, datos_y, marker='o', color=kwargs.get('color', '#e74c3c'))
        plt.xticks(rotation=45)
    elif tipo_grafico == 'pie':
        ax.pie(datos_y, labels=datos_x, autopct='%1.1f%%', shadow=True)
    elif tipo_grafico == 'barh':
        ax.barh(datos_x, datos_y, color=kwargs.get('color', '#4db6ac'))
        labelsize = kwargs.get('labelsize', None)
        if labelsize:
            ax.tick_params(axis='y', labelsize=labelsize)

    ax.set_title(titulo)
    if tipo_grafico != "pie":
        ax.set_xlabel(x_etiqueta)
        ax.set_ylabel(y_etiqueta)

    fig.tight_layout()
    plt.show(fig)

## Mostrando cantidad de seguidores por artista

In [ ]:
query_seguidores = """
MATCH (a:Artista)
OPTIONAL MATCH (u:Usuario)-[:SIGUE]->(a)
RETURN a.nombre AS artista, count(u) AS seguidores
ORDER BY seguidores DESC, artista
"""

with neo4j_driver.session() as session:
    res = session.run(query_seguidores).data()

for r in res:
    print(f"  {r['artista']}: {r['seguidores']} seguidores")

artistas_x = [r["artista"] for r in res]
seguidores_y = [r["seguidores"] for r in res]

generar_grafico_generico(artistas_x, seguidores_y, "barh","Seguidores por artista", "Seguidores", "Artista")

## Cantidad de búsquedas por género.

In [ ]:
#Provisorio
busq = redis_client.zrevrange("busquedas:genero", 0, -1, withscores=True)
generos = [g for g, _ in busq]
cantidades = [int(c) for _, c in busq]

for g, c in zip(generos, cantidades):
    print(f"{g}: {c} búsquedas")

generar_grafico_generico(generos, cantidades, "bar","Búsquedas por género", "Género", "Cantidad")

## Entradas vendidas por lugar.

In [ ]:
pipeline_recaudacion = [
    {"$lookup": {"from": "compras", "localField": "evento_id","foreignField": "evento_id","pipeline": [{"$match": {"estado": "confirmada"}}],"as": "cs"}},
    {"$project": {"_id": 0, "evento": "$nombre","recaudacion": {"$sum": "$cs.total"}}},
    {"$sort": {"recaudacion": -1, "evento": 1}
    }]

resultados = list(db_mongo.eventos.aggregate(pipeline_recaudacion))
nombres_eventos = [r["evento"] for r in resultados]
recaudacion = [r["recaudacion"] for r in resultados]

print("Recaudación por evento (compras confirmadas):")
for r in resultados:
    print(f"  {r['evento']}: ${r['recaudacion']:,}")

generar_grafico_generico(nombres_eventos, recaudacion, "barh","Recaudación por evento", "Recaudación ($)", "Evento")

## Entradas vendidas por lugar

In [ ]:
pipeline_lugares = [
    {"$match": {"estado": "confirmada"}},
    {"$lookup": {"from": "eventos","localField": "evento_id","foreignField": "evento_id","as": "ev"}},
    {"$unwind": "$ev"},         
    {"$group": {"_id": "$ev.lugar_id", "cantidad": {"$sum": "$cantidad"}}}
]

vendidas_lugar = {r["_id"]: r["cantidad"] for r in db_mongo.compras.aggregate(pipeline_lugares)}

nombres_lugares = []
entradas = []
for l in db_mongo.lugares.find().sort("lugar_id", 1):  
    cant = vendidas_lugar.get(l["lugar_id"], 0)
    nombres_lugares.append(l["nombre"])
    entradas.append(cant)
    print(f"  {l['nombre']} ({l['ciudad']}): {cant} entradas")

generar_grafico_generico(nombres_lugares, entradas, "pie", "Entradas vendidas por lugar")